## 这里我们选择使用Sqlite作为存储方案，首先需要安装langgraph-checkpoint-sqlite依赖：
uv add langgraph-checkpoint-sqlite
接着，按照以下步骤使用：
- 导入依赖
- 初始化checkpointer
- 自动建表
- 创建Agent，指定checkpointer

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

In [1]:
import os
import sqlite3
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.prebuilt import create_react_agent

# 加载环境变量
load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

# 1. 初始化大模型 (通过OpenAI兼容模式调用百炼)
model = ChatOpenAI(
    model="qwen3.6-plus",
    openai_api_key=api_key,
    openai_api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.1
)

# 2. 连接 sqlite 并初始化 checkpointer
os.makedirs("resources", exist_ok=True)
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

# 3. 创建带有记忆的 Agent
agent = create_react_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer
)

# 4. 配置 thread_id，用于区分不同用户的对话上下文
config = {"configurable": {"thread_id": "thread_user_liang"}}

# 5. 第一轮对话
print("--- 第一轮对话 ---")
response1 = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫梁哥，我最喜欢猫。")]},
    config=config
)
print(response1["messages"][-1].content)

# 6. 第二轮对话，测试记忆功能
print("\n--- 第二轮对话 (测试记忆) ---")
response2 = agent.invoke(
    {"messages": [HumanMessage(content="你还记得我叫什么名字吗？我喜欢什么动物？")]},
    config=config
)
print(response2["messages"][-1].content)

C:\Users\I\AppData\Local\Temp\ipykernel_19132\629527127.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


--- 第一轮对话 ---
你好，梁哥！很高兴认识你～🐱 猫咪确实是非常治愈又可爱的小动物。你平时是自己养猫，还是更喜欢“云吸猫”呀？有没有特别偏爱的品种，或者和猫咪之间有趣的小故事？随时欢迎跟我聊聊，我也可以帮你分享养猫小知识、猫咪冷知识，或者一起“云吸猫”！

--- 第二轮对话 (测试记忆) ---
当然记得啦！你叫**梁哥**，最喜欢的动物是**猫**～🐱  
需要聊聊你家猫主子的日常、分享养猫小技巧，还是单纯想“云吸猫”？随时告诉我！
